# 감정 모델 추가 학습 (KEMDy + AI Hub 4차년도)
`HyunaZ/hubert_emotion`을 이어서 학습합니다. **런타임 → 런타임 유형 변경 → GPU(T4)** 로 먼저 바꾸세요.

- **KEMDy**(ETRI 대회 저장소의 일부, 라벨 약 2,400개)는 공개 저장소에서 바로 받습니다.
- **AI Hub 4차년도**(14,606개)는 직접 Drive에 올려야 합니다. 내 Drive의 `aihub` 폴더에 `4차년도.zip`과 `4차년도.csv`를 올려 두세요.
- AI Hub 4차년도에는 행복·놀람이 거의 없어서, KEMDy와 **섞어서** 학습하고 **KEMDy 세션 3(학습에 안 쓴 데이터)** 으로 평가합니다.

> **AI Hub 약관 안내**: 이 데이터는 학습용으로만 쓸 수 있고, 국외 서버로의 반출(2·3항)에는 별도 합의가 필요하며, 이용한 모델과 앱에는 "한국지능정보사회진흥원의 사업결과"임을 밝혀야 합니다. Colab과 Drive는 해외 서버이므로 이 노트북 실행에 따른 책임은 실행하는 사람에게 있습니다(약관 6항). 상업적 이용은 별도 협의가 필요합니다.

In [ ]:
# 1) GPU 확인 — 'Tesla T4' 같은 표가 나와야 합니다
!nvidia-smi

In [ ]:
# 2) 코드 받기 (공개 저장소)
!git clone https://github.com/jaeuk6797-cell/kuccton.git
%cd kuccton
!git checkout feat/finetune-kemdy19

In [ ]:
# 3) 라이브러리 설치 (torch는 Colab에 이미 있습니다)
!pip -q install transformers pytorch-lightning librosa huggingface_hub

In [ ]:
# 4) KEMDy 데이터 받기 (공개 저장소, 약 1GB, 몇 분 걸립니다)
!git clone --depth 1 https://github.com/SeolRoh/Multi-Still_ETRI.git /content/KEMDy

In [ ]:
# 5) AI Hub 4차년도 가져오기: Drive 연결 허용 창이 뜹니다. 큰 파일이라 몇 분 걸립니다.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/aihub
!cp "/content/drive/MyDrive/aihub/4차년도.zip" /content/aihub/
!cp "/content/drive/MyDrive/aihub/4차년도.csv" /content/aihub/
!ls -la /content/aihub

In [ ]:
# 6) 학습 전에 데이터 확인: KEMDy 라벨 수(약 2,400)와 AI Hub 라벨 수(4차년도 약 13,500)가 나옵니다
!python training/finetune.py --data /content/KEMDy --aihub-zip /content/aihub/4차년도.zip --aihub-csv /content/aihub/4차년도.csv --inspect

In [ ]:
# 7) 학습. 지난번 결과(중립으로만 쏠림)를 고치는 설정입니다.
#    --merge-negative : 혐오·공포를 분노로 합침(앱도 '긴장' 하나로 묶어 씀)
#    --balance sampler: 소수 클래스를 더 자주 뽑음 / --cap-per-class: 한 감정이 너무 많지 않게 제한
# 출력의 '[시작 전]'과 '[에폭 N]'을 비교하세요. 마지막 줄에 '쓸 만함' 또는 '개선 없음'이 나옵니다.
!python training/finetune.py --data /content/KEMDy --aihub-zip /content/aihub/4차년도.zip --aihub-csv /content/aihub/4차년도.csv --out /content/out --epochs 4 --merge-negative --balance sampler --cap-per-class 1500

In [ ]:
# 8) 결과를 Drive에 저장 (런타임이 끊겨도 남도록)
!cp /content/out/hubert_emotion_kemdy19.ckpt /content/drive/MyDrive/hubert_emotion_aihub4_kemdy.ckpt
!ls -la /content/drive/MyDrive/hubert_emotion_aihub4_kemdy.ckpt

## 끝난 뒤
1. 7번 칸 마지막 줄이 **'쓸 만함'** 일 때만 `hubert_emotion_aihub4_kemdy.ckpt`를 PC로 내려받아 쓰세요. **'개선 없음'** 이면 적용하지 마세요.
2. 서버는 `EMOTION_CKPT` 환경변수로 그 파일 경로를 알려 주면 새 모델을 씁니다(server 폴더에서 `python -m uvicorn server:app --port 8000`).
3. 평가는 학습에 쓰지 않은 KEMDy 세션 3으로 합니다. 앱에서 쓰는 실제 목소리와는 다를 수 있으니, 점수가 올라도 직접 녹음해서 확인하세요.